# Manga / Webtoon → Dialogue Text (Colab)

파이프라인: `PDF/이미지 → Koharu RF-DETR → OCR → 언어 판별 → 외국어만 Qwen 번역 → JSONL/TXT`

기본값은 일본 만화용 `MangaOCR + RTL`입니다. 한국 웹툰은 설정 셀에서 `OCR_BACKEND="paddle"`, `PADDLE_LANG="korean"`, `READING_DIRECTION="ltr"`로 바꾸면 됩니다.

## 1. 패키지 설치 + 저장소 가져오기

In [ ]:
!pip -q install \
    "rfdetr==1.7.0" \
    "safetensors>=0.5" \
    "huggingface_hub>=0.27" \
    "manga-ocr>=0.1.11" \
    "paddlepaddle>=3.0" \
    "paddleocr>=3.0" \
    "transformers>=4.51" \
    "accelerate>=1.2" \
    "bitsandbytes>=0.45" \
    "lingua-language-detector>=2.0" \
    "pymupdf>=1.24" \
    pillow numpy tqdm

!rm -rf /content/manga2text_tmp
!git clone -q https://github.com/HisameOgasahara/manga2text_tmp.git /content/manga2text_tmp

## 2. 설정

여기만 바꾸면 대부분의 동작을 조절할 수 있습니다.

In [ ]:
import sys
from pathlib import Path

sys.path.append("/content/manga2text_tmp")

from manga2text_pipeline import (
    build_language_detector,
    collect_page_images,
    load_koharu_detector,
    load_ocr_backend,
    load_translation_model,
    process_pages,
    save_results,
)

WORK_DIR = Path("/content/manga2text")
INPUT_DIR = WORK_DIR / "input"
PAGE_DIR = WORK_DIR / "pages"
OUTPUT_DIR = WORK_DIR / "output"

for directory in [INPUT_DIR, PAGE_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# OCR: "manga" 또는 "paddle"
OCR_BACKEND = "manga"

# PaddleOCR 사용 시: "korean", "japan", "ch", "en" 등
PADDLE_LANG = "korean"
PADDLE_DEVICE = "cpu"

# 일본 만화: "rtl", 한국 웹툰/영문 코믹: "ltr"
READING_DIRECTION = "rtl"
ROW_TOLERANCE = 80

# 효과음까지 OCR하려면 True
INCLUDE_SFX = False
CROP_PADDING = 8

CLASS_THRESHOLDS = {
    0: 0.25,  # text
    1: 0.20,  # onomatopoeia / SFX
    2: 0.50,  # bubble
    3: 0.50,  # panel
}

ENABLE_TRANSLATION = True
TRANSLATION_MODEL = "Qwen/Qwen3-1.7B"
# TRANSLATION_MODEL = "Qwen/Qwen3-4B"
MAX_NEW_TOKENS = 256

PDF_DPI = 200
PAGE_LIMIT = None  # 예: 10이면 앞 10페이지만 테스트

## 3. PDF / 이미지 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()

for filename, data in uploaded.items():
    destination = INPUT_DIR / filename
    destination.write_bytes(data)

print("업로드 완료")
for path in sorted(INPUT_DIR.iterdir()):
    print(" -", path.name)

## 4. PDF → 페이지 이미지 변환

In [ ]:
page_paths = collect_page_images(
    input_dir=INPUT_DIR,
    page_dir=PAGE_DIR,
    pdf_dpi=PDF_DPI,
    page_limit=PAGE_LIMIT,
)

print(f"처리할 페이지 수: {len(page_paths)}")
for path in page_paths[:10]:
    print(" -", path)

## 5. Koharu RF-DETR 다운로드 + 로드

공개 모델 `mayocream/koharu-layout-rfdetr-seg-2xl-1152`의 `model.safetensors`와 공식 loader를 Hugging Face에서 자동 다운로드합니다.

In [ ]:
import torch

detector = load_koharu_detector()

print("RF-DETR 로드 완료")
print("CUDA 사용 가능:", torch.cuda.is_available())

## 6. OCR 모델 로드

`OCR_BACKEND="manga"`이면 MangaOCR, `"paddle"`이면 PaddleOCR를 사용합니다.

In [ ]:
ocr_model = load_ocr_backend(
    backend=OCR_BACKEND,
    paddle_lang=PADDLE_LANG,
    paddle_device=PADDLE_DEVICE,
)

print("OCR 로드 완료:", OCR_BACKEND)

## 7. 언어 판별기 로드

한글/가나 문자 범위를 먼저 확인하고, 애매한 경우 Lingua로 한국어·일본어·중국어·영어를 판별합니다.

In [ ]:
language_detector, language_to_code = build_language_detector()
print("언어 판별기 로드 완료")

## 8. 소형 번역 LLM 로드

기본은 `Qwen3-1.7B` 4-bit입니다. 한국어로 판별된 텍스트는 번역 단계를 건너뜁니다.

In [ ]:
translation_tokenizer = None
translation_model = None

if ENABLE_TRANSLATION:
    translation_tokenizer, translation_model = load_translation_model(
        model_name=TRANSLATION_MODEL,
    )
    print("번역 모델 로드 완료:", TRANSLATION_MODEL)
else:
    print("번역 비활성화")

## 9. 전체 파이프라인 실행

페이지별로 `RF-DETR 검출 → 읽기 순서 근사 → OCR → 언어 판별 → 외국어만 한국어 번역`을 수행합니다.

In [ ]:
records = process_pages(
    page_paths=page_paths,
    detector=detector,
    ocr_backend=OCR_BACKEND,
    ocr_model=ocr_model,
    language_detector=language_detector,
    language_to_code=language_to_code,
    class_thresholds=CLASS_THRESHOLDS,
    reading_direction=READING_DIRECTION,
    row_tolerance=ROW_TOLERANCE,
    crop_padding=CROP_PADDING,
    include_sfx=INCLUDE_SFX,
    enable_translation=ENABLE_TRANSLATION,
    translation_tokenizer=translation_tokenizer,
    translation_model=translation_model,
    max_new_tokens=MAX_NEW_TOKENS,
)

print(f"추출된 텍스트 영역: {len(records)}")

## 10. 결과 미리보기

In [ ]:
for record in records[:30]:
    print(
        f"[p.{record['page']:03d} / {record['order']:02d}] "
        f"{record['language']} | "
        f"{record['original']} "
        f"-> {record['korean']}"
    )

## 11. JSONL / TXT 저장 + 다운로드

In [ ]:
jsonl_path, txt_path = save_results(
    records=records,
    output_dir=OUTPUT_DIR,
)

print("저장 완료")
print(" -", jsonl_path)
print(" -", txt_path)

files.download(str(jsonl_path))
files.download(str(txt_path))